In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os
from pathlib import Path
import sys
from bisect import bisect_left, bisect_right
from dataclasses import asdict, dataclass
from typing import Callable
from datetime import datetime
from contextlib import contextmanager
import time
import uuid

root_dir = Path(os.getcwd()).parent
src_dir = root_dir / 'ingestion/src'

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from helpers.pdfparser import PdfParser, MarkdownSection, SectionMetadata
from helpers.indexer import Indexer, IndexDocument, IndexChunk
from helpers.hasher import DocumentHasher
from helpers.dbrepository import FileRepository


os.environ['NLTK_DATA'] = str(root_dir / '.nltk_data')
os.environ['QDRANT_URL'] = 'http://localhost:6333'

embedding_cache_dir = str(root_dir / 'models/embeddings')

In [3]:
def enrich_chunk_with_citation_data(chunk: IndexChunk, metadata: dict, page_offsets: list) -> IndexChunk:

    if chunk and metadata:
        section_start_idx = metadata['section_start_char_idx'] 

        chunk_start_idx = section_start_idx + chunk.start_char_idx
        chunk_end_idx = chunk_start_idx + len(chunk.text) 
    
        chunk_start_page_idx = bisect_right(page_offsets, chunk_start_idx) - 1
        chunk_end_page_idx = bisect_left(page_offsets, chunk_end_idx) - 1

        #set citation data in the chunk object
        chunk.start_char_idx = chunk_start_idx
        chunk.end_char_idx = chunk_end_idx
        chunk.start_page_idx = chunk_start_page_idx
        chunk.end_page_idx = chunk_end_page_idx

    return chunk 

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        end = time.perf_counter()
        print(f"{label}: {end - start:.6f} seconds")


In [ ]:
hasher = DocumentHasher()
file_repository = FileRepository(db_path=str(root_dir / 'database/rag.db'))

file_names = ['2505.07891.pdf']

for file_name in file_names:
    print(f'------------ processing {file_name}')
    file_id = str(uuid.uuid4().hex) 
    file_path = root_dir / f'staging/{file_name}'
    file_hash = hasher.hash_document(file_path)
    
    with timer('parsing completed..'):
        parser = PdfParser()
        file_metadata, page_offsets, sections = parser.parse(file_path)    

    with timer('prepare indexing documents..'):
        indexer = Indexer(qdrant_url='http://localhost:6333', collection_name='hybrid_collection', fast_embedding_name='BAAI/bge-small-en-v1.5')

        #create index_docs
        index_docs = [
                IndexDocument(    
                    doc_id = f'{file_hash}_{hasher.hash_document(section.text.encode())}',
                    text=section.text,
                    metadata=section.metadata,
                ) for section in sections if isinstance(section, MarkdownSection)]

    with timer('indexing completed..'):                                         
        #indexer chunks the documents and indexes them in the vector store.
        index = indexer.index(
            index_docs, 
            transform_chunk_fn=lambda c, m: enrich_chunk_with_citation_data(c, m, page_offsets),
            )

    with timer('state updated in DB..'):
        file_repository.insert_or_ignore(
            file_id=file_id,
            content_hash=file_hash,
            content_length=file_metadata.get('file_length', 0),
            metadata=file_metadata
        )
    

In [7]:
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.embeddings.fastembed import FastEmbedEmbedding

#initialize local embedding model
local_embed = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5", cache_dir=embedding_cache_dir)
Settings.embed_model = local_embed


Ignoring corrupted tree cache file /home/chins/git/rag-architecture/models/embeddings/models--qdrant--bge-small-en-v1.5-onnx-q/trees/52398278842ec682c6f32300af41344b1c0b0bb2.json: [Errno 13] Permission denied: '/home/chins/git/rag-architecture/models/embeddings/models--qdrant--bge-small-en-v1.5-onnx-q/trees/52398278842ec682c6f32300af41344b1c0b0bb2.json'


In [8]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
import qdrant_client

# Initialize the Qdrant client
client = qdrant_client.QdrantClient(url="http://localhost:6333")

#initiaalize qdrant vector store
# vector_store = QdrantVectorStore(
#     client=client, 
#     collection_name="test_collection"
#     )

#initialize qdrant vector store with hybrid search enabled
vector_store = QdrantVectorStore(
    client=client, 
    collection_name="hybrid_collection",
    enable_hybrid=True,
    batch_size = 64,
    )

#initialize storage_context over the vector storage
storage_context = StorageContext.from_defaults(vector_store=vector_store)


Fetching 5 files: 100%|██████████| 5/5 [02:29<00:00, 29.90s/it]


In [ ]:

async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about individuals"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "What are the differences between the play and the drama?"
    )
    print(response)

await main()

In [ ]:
client.retrieve('test_collection', ids=['0001dc07-2689-4944-adfb-96f8dbc7eb46'])

In [ ]:
from qdrant_client.http import models


#nodes_to_check = [node for node in nodes]
node_ids_to_check = ['0001dc07-2689-4944-adfb-96f8dbc7eb46']

scroll_results, _ = client.scroll(
    collection_name="test_collection",
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="chunk_id", # LlamaIndex maps node_id to the 'id' key in payload
                match=models.MatchAny(any=node_ids_to_check),
            )
        ]
    ),
    with_vectors=False,
    with_payload=True, # We need the payload to read the custom string ID
    limit=len(node_ids_to_check),
)

# 3. Extract the found IDs from the payload
existing_ids = {point.payload["chunk_id"] for point in scroll_results if "chunk_id" in point.payload}

existing_ids

In [ ]:
# import pymupdf4llm
# from markdown_it import MarkdownIt
# from llama_index.core import Document

# #read full_text as well as
# pages = pymupdf4llm.to_markdown('../staging/2505.07891.pdf', page_chunks=True, force_ocr=False)
# full_md = ''.join([page['text'] for page in pages])


In [ ]:
import pymupdf4llm 
pymupdf4llm.use_layout(False)
pages = pymupdf4llm.to_markdown('../staging/OTC_TCS_2025.pdf', page_chunks=True,use_ocr=False)
pages = [page for page in pages]

Warning - arguments ignored in legacy mode: {'use_ocr'}.


In [24]:
full_md = ''.join([page['text'] for page in pages])

In [30]:
pages[63]['text']

'**Notice** **62**\n\n\n**Annexure A**\n\n\n**Details of Director seeking appointment and re-appointment at this AGM**\n\n\n**(In pursuance of Regulation 36(3) of the SEBI Listing Regulations and Secretarial Standard - 2 on General Meetings)**\n\n\n**<u>Name of the Director</u>** **<u>Aarthi Subramanian</u>**\n\n**<u>Director Identification Number (DIN)</u>** <u>07121802</u>\n\n**<u>Designation and Category of Director</u>** <u>Executive Director - President and Chief Operating Officer</u>\n\n**<u>Date of birth and age</u>** <u>June 26, 1967 (57 years)</u>\n\n**Date of first appointment** March 12, 2015\n\n\n(Aarthi Subramanian was appointed as an Executive Director on March 12, 2015 and later as a\n<u>Non-Executive Director on August 17, 2017)</u>\n\n**Qualifications** \uf09f Bachelor of Technology in Computer Science\n<u>\uf09f</u> <u>Master’s Degree in Engineering Management</u>\n\n**Brief profile** Aarthi Subramanian is the Executive Director – President and Chief Operating Officer

In [ ]:
# file_metadata, page_offsets = get_pdf_metadata(pages)

# #parse the markdown to find logical sections (treating h2 as the cut_level)
# hierarchical_sections = markdown_sections(full_md, page_offsets, file_metadata)
# sections = get_flat_sections(hierarchical_sections)

In [ ]:
# for chunk in chunks:
#     print(f'---------------------------------------------------------')
    
#     print(f'chunk span:{chunk.chunk_start_char_idx}-{chunk.chunk_end_char_idx}, pages:{chunk.chunk_start_page}-{chunk.chunk_end_page}')
#     print(f'parent section span:{chunk.section['section_start_char_idx']}-{chunk.section['section_end_char_idx']}')


In [ ]:
# from llama_index.core.schema import NodeRelationship
# node_idx = 10

# chunk = nodes[node_idx]
# source_doc = chunk.relationships[NodeRelationship.SOURCE]
# chunk_start_idx, chunk_end_idx = chunk.start_char_idx, chunk.end_char_idx

# print(source_doc.metadata['start_page'])
# print(chunk_start_idx, chunk_end_idx)

In [9]:
from pydantic import BaseModel, Field

class Citation(BaseModel):
    ''' A citation source'''
    id: int = Field(description="source #")

class AnswerPart(BaseModel):
    '''Answer part backed by citations'''
    part: str =  Field(description="The answer part text")
    citations: list[Citation] = Field(description="A list of citations backing the answer part")

class RAGResponse(BaseModel):
    '''Citation backed answer. An Answer consists of one or more parts'''
    parts: list[AnswerPart] = Field(description="A list of answer parts")


In [ ]:
vector_store = QdrantVectorStore(
    client=client, 
    collection_name="hybrid_collection",
    enable_hybrid=True,
    batch_size = 64,
    )



In [ ]:
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.core import PromptTemplate
from llama_index.llms.openai import OpenAI

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

custom_citation_prompt = PromptTemplate(
    "Please answer the query based on the provided context.\n"
    "Every time you use a source text to generate your answer part, you must add the corresponding citation source.\n"
    
    "Do not add explicit references in square brackets in your answer. Instead use the citations in the structured output resposne."
    "Do not use external knowledge. If you do not know the answer, send empty response.\n\n"
    "Context:\n{context_str}\n\n"
    "Query: {query_str}\n\n"
    "Answer:"
)

llm = OpenAI(model="gpt-4o-mini").as_structured_llm(RAGResponse)

query_engine = CitationQueryEngine.from_args(
    index,
    llm=llm,
    citation_chunk_size=512,       # Granular text blocks for inline citations
    citation_chunk_overlap=20,     # Small overlap to prevent cut-off quotes
    similarity_top_k=5,            # Number of source nodes to retrieve
    citation_qa_template=custom_citation_prompt,
)

retriever = index.as_retriever()

In [ ]:
response = query_engine.query('Explain the topic specific text-rank in short. Also, what is the benefit of TrumourGPT?')
response

In [39]:
results = retriever.retrieve('Date of birth and age of Aarthi Subramanian and also her education details.')
for result in results:
    print(result.node.text)

As Executive Director and Global Head of Delivery Excellence,<br>Governance & Compliance at TCS, she was responsible for ensuring service delivery excellence,<br>governance of large transformation programs, and overseeing risk and compliance across<br>global operations.  As Head of Delivery for Retail & CPG Business Unit at TCS, she led strategic<br>accounts and major client relationships, driving customer satisfaction through rigorous service<br>delivery and value addition.<br>Aarthi Subramanian holds a Bachelor of Technology in Computer Science from the National<br>Institute of Technology, Warangal, India, and a Master’s degree in Engineering Management<br>from the University of Kansas, USA.<br>Her exemplary contributions have been acknowledged through various accolades such<br>as “Technology Leader of the Year” at ETPrime Women Leadership Awards (2019) where<br>she was honoured for her pioneering efforts in driving technological innovation and digital<br>transformation within the Ta

In [36]:
response = query_engine.query('Date of birth and age of Aarthi Subramanian')
response


PydanticResponse(response=RAGResponse(parts=[AnswerPart(part='The date of birth of Aarthi Subramanian is not explicitly mentioned in the provided context, and therefore, her age cannot be determined from the available information.', citations=[Citation(id=3), Citation(id=4)])]), source_nodes=[NodeWithScore(node=TextNode(id_='243915b0-868b-4188-9bf9-33b49ad9b25c', embedding=None, metadata={'section_title': '**Q2**', 'section_start_char_idx': 29304, 'section_end_char_idx': 212985, 'file_name': 'OTC_TCS_2025_171a0be26cc6475989242422405eabe6.pdf', 'file_path': '/app/staging/fetched/OTC_TCS_2025_171a0be26cc6475989242422405eabe6.pdf', 'file_title': 'TCS Integrated Annual Report 2024-25 (rev)', 'file_author': '', 'file_length': 1127869, 'file_total_pages': '', 'chunk_id': '1f437fa9343a88a38de59f8fb4f8b8b142110739e696c50b730f93d014a09872_d172dbb66abb73eb2f6c979a7de15a43abdb06bbb3fa3105d76f6abcc4a8bfc5_chunk_37', 'chunk_start_char_idx': 29304, 'chunk_end_char_idx': 33333, 'chunk_start_page_idx'

In [22]:
for source in response.source_nodes:
    print(source.node.text)


Source 1:
As Executive Director and Global Head of Delivery Excellence,<br>Governance & Compliance at TCS, she was responsible for ensuring service delivery excellence,<br>governance of large transformation programs, and overseeing risk and compliance across<br>global operations.  As Head of Delivery for Retail & CPG Business Unit at TCS, she led strategic<br>accounts and major client relationships, driving customer satisfaction through rigorous service<br>delivery and value addition.<br>Aarthi Subramanian holds a Bachelor of Technology in Computer Science from the National<br>Institute of Technology, Warangal, India, and a Master’s degree in Engineering Management<br>from the University of Kansas, USA.<br>Her exemplary contributions have been acknowledged through various accolades such<br>as “Technology Leader of the Year” at ETPrime Women Leadership Awards (2019) where<br>she was honoured for her pioneering efforts in driving technological innovation and digital<br>transformation wit

In [ ]:
# from llama_index.core.schema import NodeRelationship
# sources = response.source_nodes

# text1 = sources[0].node.text[10:]
# text2 = sources[1].node.text[10:]

# source_chunk_id = sources[0].metadata['chunk_id']
# source_node = index.docstore.get_node(source_chunk_id)

# print(f'node text: {source_node.text[:100]}......{source_node.text[-100:]}')

# print(f'text1 start: {text1[:100]}')
# print(f'text2 end  : {text2[-100:]}')

# len(text1 + text2), len(source_node.text)


In [ ]:
async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about documents"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""
    You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. For provenance, generate citation which includes the details about the source file name, section name, and the psource age(s) 
    which your answers are based on. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "Explain the topic specific text-rank. Also, what are the three techniques used by Trurumour to verify the truthfulness of the health news?"
    )
    print(response)

await main()

In [ ]:
import sqlite3

db_path = "../database/rag.db"

def initialize_db(db_path):
    with sqlite3.connect(db_path) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                document_id TEXT PRIMARY KEY,
                content_hash TEXT NOT NULL UNIQUE,
                file_size INTEGER NOT NULL,
                file_name TEXT NOT NULL,
                file_path TEXT NOT NULL,
                created_at TEXT NOT NULL,
                metadata TEXT
            )
        """)

initialize_db(db_path)

In [ ]:
class DocumentRepository:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.create_schema(db_path)

    def create_schema(self, db_path):
        with sqlite3.connect(db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS documents (
                    document_id TEXT PRIMARY KEY,
                    content_hash TEXT NOT NULL UNIQUE,
                    file_size INTEGER NOT NULL,
                    file_name TEXT NOT NULL,
                    file_path TEXT NOT NULL,
                    metadata TEXT,
                    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)

    def find_by_hash(self, content_hash):
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute(
                """
                SELECT document_id, file_size
                FROM documents
                WHERE content_hash = ?
                """,
                (content_hash,),
            ).fetchone()

    def insert_or_ignore(self, document_id, content_hash, file_size, file_name, file_path, metadata='{}'):
        ret = None
        with sqlite3.connect(self.db_path) as conn:
            
            ret = conn.execute(
                """
                INSERT OR IGNORE INTO documents
                    (document_id, content_hash, file_size, file_name, file_path, metadata)
                VALUES (?, ?, ?, ?, ?, ?)
                RETURNING document_id;
                """,
                (document_id, content_hash, file_size, file_name, file_path, metadata),
            ).fetchone()
        print(ret)
        return ret

In [ ]:
repository = DocumentRepository(db_path)

repository.insert_or_ignore('123', 'abc', 100, 'agas.txt', '../staging/agas.txt')
repository.insert_or_ignore('124', 'xyz', 200, 'ads.txt', 'http://foo.bar.com/ads.txt')


In [ ]:
repository.find_by_hash('xyzc')

In [ ]:
repository.insert_or_ignore('13ss2', 'abcfs', 100, 'agas.txt', '../staging/agas.txt')

In [ ]:
import hashlib
m = hashlib.sha256()
m.update(b"Nobody inspects")
m.update(b" the spammish repetition")

print(m.hexdigest())

m2 = hashlib.sha256()
m2.update(b"Nobody inspects the spammish repetition")
print(m2.hexdigest())

In [ ]:
arr = ['foo', 'bar']
for i, ele in enumerate(arr):
    print(i, ele)